# 20. PCA 기반 이상치 탐지 (PCA Anomaly Detection)

2차원 데이터를 `PCA(n_components=1)`로 압축한 뒤 `inverse_transform`으로 복원하여 원본과의 재구성 오차(Reconstruction Error)가 큰 샘플을 이상치로 탐지합니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
normal_data = np.random.normal(loc=0, scale=1, size=(100, 2))
anomalies = np.random.normal(loc=5, scale=5, size=(5, 2))
data = np.vstack([normal_data, anomalies])
data_scaled = StandardScaler().fit_transform(data)

## 문제 1. PCA 압축·복원 및 재구성 오차 계산하기

`PCA(n_components=1, random_state=42)`로 `data_scaled`를 변환하고 `inverse_transform`으로 복원한 뒤 샘플별 재구성 오차(제곱합)의 상위 5개 값을 출력하세요.

In [ ]:
pca = PCA(n_components=1, random_state=42)
data_pca = pca.fit_transform(data_scaled)
data_rec = pca.inverse_transform(data_pca)
rec_error = np.sum((data_scaled - data_rec) ** 2, axis=1)
print('상위 5개 재구성 오차:', np.round(np.sort(rec_error)[-5:], 4))

**결과 해석**

주성분 직선으로 압축 후 복원할 때 정상 패턴에서 벗어난 이상치일수록 주성분 축과 거리가 멀어 재구성 오차가 크게 나타납니다.

## 문제 2. 95백분위 임계치 기반 이상치 시각화하기

재구성 오차의 상위 95백분위(`np.percentile(rec_error, 95)`)를 초과하는 샘플을 빨간색 `x` 마커로 구분해 산점도를 그리세요.

In [ ]:
pca = PCA(n_components=1, random_state=42)
data_rec = pca.inverse_transform(pca.fit_transform(data_scaled))
rec_error = np.sum((data_scaled - data_rec) ** 2, axis=1)

threshold = np.percentile(rec_error, 95)
is_anomaly = rec_error > threshold
print(f'임계치: {threshold:.4f}, 탐지 수: {is_anomaly.sum()}')

plt.figure(figsize=(7, 5))
plt.scatter(data_scaled[~is_anomaly, 0], data_scaled[~is_anomaly, 1], label='Normal', alpha=0.7)
plt.scatter(data_scaled[is_anomaly, 0], data_scaled[is_anomaly, 1], color='red', marker='x', s=90, label='Anomaly')
plt.legend()
plt.title('Detected Anomalies via PCA')
plt.show()

**결과 해석**

상위 5% 재구성 오차 임계치를 기준으로 정상 군집 외곽에 위치한 이상치 샘플을 비지도 방식으로 효과적으로 분리할 수 있습니다.